In [ ]:
# ---
# Plotting Pairwise IBD (PI_HAT) Between Samples
# Visualizations include heatmaps and median IBD distributions
# ---

In [ ]:


import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Configuration
ibd_file = "analysis_results/variants_analysis/files/ibd_pm.genome"
output_dir = "analysis_results/variants_analysis/plots/"

# Load IBD Data
ibd = pd.read_csv(ibd_file, sep=r"\s+")

# Clean sample names (extract consistent IDs)
ibd["IID1"] = ibd["IID1"].str.extract(r'-(MD\d+)-?')
ibd["IID2"] = ibd["IID2"].str.extract(r'-(MD\d+)-?')

# Build IBD Matrix
samples = pd.unique(ibd[['IID1', 'IID2']].values.ravel())
ibd_matrix = pd.DataFrame(0.0, index=samples, columns=samples)

# Fill IBD matrix with PI_HAT values
for _, row in ibd.iterrows():
    ibd_matrix.loc[row['IID1'], row['IID2']] = row['PI_HAT']
    ibd_matrix.loc[row['IID2'], row['IID1']] = row['PI_HAT']

# Heatmap: All PI_HAT values
plt.figure(figsize=(10, 8))
sns.heatmap(ibd_matrix, cmap="YlGnBu", square=True, xticklabels=True, yticklabels=True)
plt.title("Pairwise IBD (PI_HAT) Heatmap (Pm)")
plt.tight_layout()
plt.savefig(output_dir/"Pairwise_IBD_Heatmap.png", dpi=300, bbox_inches='tight')
plt.show()


# Heatmap: Filtered (PI_HAT > 0)
np.fill_diagonal(ibd_matrix.values, np.nan)
ibd_matrix_cleaned = ibd_matrix.replace(0, np.nan).dropna(how='all', axis=0).dropna(how='all', axis=1)

plt.figure(figsize=(10, 8))
sns.heatmap(ibd_matrix_cleaned, cmap="YlGnBu", square=True, xticklabels=True, yticklabels=True, annot=True)
plt.title("Pairwise IBD Heatmap (PI_HAT > 0) (Pm)")
plt.tight_layout()
plt.savefig(output_dir/"Pairwise_IBD_Heatmap>0.png", dpi=300, bbox_inches='tight')
plt.show()

# Median IBD per Sample
median_ibd = ibd_matrix.replace(0, np.nan).median(axis=1)

plt.figure(figsize=(12, 4))
sns.barplot(x=median_ibd.index, y=median_ibd.values, palette="mako", hue=median_ibd.index, legend=False)
plt.xticks(rotation=90)
plt.ylabel("Median PI_HAT")
plt.title("Median Pairwise IBD per Sample (Pm)")
plt.tight_layout()
plt.savefig(output_dir/"Median_Pairwise_IBD_per_Sample.png", dpi=300, bbox_inches='tight')
plt.show()
